# Day 04 – Perturbation Modelling Lite

We pretend some cells were treated vs. control and train a tiny classifier + run DEGs to spot the response.


### What happens today?
1. Load annotated data, then add a fake `condition` column.
2. Train Logistic Regression on PCA embeddings to predict the condition.
3. Run DEG analysis between control vs. perturbation cells.
4. (Bonus) If `scvi-tools` is installed, learn a latent space that respects condition labels.


In [ ]:
# Install the needed libraries once (delete the # to run)
# %pip install --quiet scanpy scvi-tools scvelo gseapy networkx


### Step 1 – Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

try:
    import scvi
except ImportError:
    scvi = None
    print('⚠️ Install scvi-tools (pip install scvi-tools) to unlock the perturbation modeling demo.')

try:
    import gseapy as gp
except ImportError:
    gp = None
    print('⚠️ Install gseapy (pip install gseapy) to run the GSEA step.')

try:
    import scvelo as scv
except ImportError:
    scv = None
    print('⚠️ Install scvelo (pip install scvelo) to run the RNA velocity step.')

import networkx as nx

sc.settings.verbosity = 0
sc.set_figure_params(dpi=100)


### Step 2 – Load annotated data

In [ ]:
SHARED_DIR = Path('..') / 'shared_data'
SHARED_DIR.mkdir(parents=True, exist_ok=True)
day2_file = SHARED_DIR / 'day02_annotated.h5ad'

print('Looking for Day 02 annotated data at', day2_file)

def build_day2_from_scratch():
    """Create the Day 02 dataset (clean + cluster + annotate) on the fly."""
    data = sc.datasets.pbmc3k()
    data.var_names_make_unique()
    data.layers['counts'] = data.X.copy()
    data.obs['source_dataset'] = 'pbmc3k'
    data.raw = data

    sc.pp.filter_cells(data, min_genes=200)
    sc.pp.filter_genes(data, min_cells=3)
    data.var['mt'] = data.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(data, qc_vars=['mt'], inplace=True)
    data = data[data.obs['pct_counts_mt'] < 15, :]

    sc.pp.normalize_total(data, target_sum=1e4)
    sc.pp.log1p(data)
    sc.pp.highly_variable_genes(data, n_top_genes=2000, subset=True)
    sc.pp.scale(data, max_value=10)

    sc.tl.pca(data, n_comps=50)
    sc.pp.neighbors(data, n_neighbors=15)
    sc.tl.umap(data)

    sc.tl.leiden(data, resolution=0.5, key_added='leiden')
    marker_map = {
        '0': 'Naive T',
        '1': 'Memory T',
        '2': 'B cell',
        '3': 'NK',
        '4': 'Myeloid',
        '5': 'Plasma',
    }
    data.obs['cell_type'] = data.obs['leiden'].map(marker_map).fillna('Other')
    data.obs['batch'] = 'Batch_A'
    return data

if day2_file.exists():
    adata_from_day2 = sc.read(day2_file)
    print('✅ Loaded annotated data from Day 02 file.')
else:
    print('⚠️ Day 02 file not found. Re-running the quick Day 02 pipeline now...')
    adata_from_day2 = build_day2_from_scratch()


In [ ]:
adata_day4 = adata_from_day2.copy()


### Step 3 – Fake a perturbation label and train a simple classifier

In [ ]:
np.random.seed(7)
adata_day4.obs['condition'] = np.random.choice(['control', 'perturbation'], size=adata_day4.n_obs, p=[0.6, 0.4])

X = adata_day4.obsm['X_pca']
y = (adata_day4.obs['condition'] == 'perturbation').astype(int).values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=7)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['control', 'perturbation']))


### Step 4 – Differential genes between conditions

In [ ]:
sc.tl.rank_genes_groups(adata_day4, groupby='condition', method='wilcoxon')
sc.pl.rank_genes_groups(adata_day4, n_genes=5, sharey=False)


### Step 5 – Bonus: learn a condition-aware latent space with scVI

In [ ]:
if scvi is None:
    print('Install scvi-tools to unlock this step.')
else:
    scvi.data.setup_anndata(adata_day4, batch_key='condition')
    vae = scvi.model.SCVI(adata_day4, n_latent=20)
    vae.train(max_epochs=30, plan_kwargs={'lr': 3e-3}, check_val_every_n_epoch=None)
    adata_day4.obsm['X_scvi'] = vae.get_latent_representation()
    print('Stored SCVI latent space in adata_day4.obsm['X_scvi']')
